In [58]:
import pandas as pd
from dotenv import load_dotenv
import numpy as np
from pandas import Timestamp

import os
import json
import requests
from datetime import datetime, timezone
import zoneinfo
import warnings
import glob
import difflib
import time


import db_functions 
from importlib import reload
reload(db_functions)
 
from db_functions import fetch_data, types_fix, bronze_builds, bronze_jobs_logs, bronze_agents, calculate_wait_time, reconstruct_log


pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
warnings.filterwarnings('ignore')

load_dotenv()

BUILDKITE_API_TOKEN = os.getenv('BUILDKITE_API_TOKEN')
GMAIL_USERNAME = os.getenv('GMAIL_USERNAME')
GMAIL_PASSWORD = os.getenv('GMAIL_PASSWORD')
ORGANIZATION_SLUG='vllm'
PIPELINE_SLUG = 'ci'#'ci-aws'
LAST_24_HOURS = (datetime.utcnow() - pd.Timedelta(hours=24)).strftime('%Y-%m-%dT%H:%M')
WAITING_TIME_ALERT_THR = 10800 # 3 hours
AGENT_FAILED_BUILDS_THR = 3 # agents declaired unhealthy if they have failed jobs from >=3 unique builds
RECIPIENTS = ['hissu.hyvarinen@amd.com', 'olga.miroshnichenko@amd.com', 'alexei.ivanov@amd.com']
PATH_TO_LOGS = '/mnt/home/buildkite_logs/db/'
PATH_TO_TABLES = '/mnt/home/vllm_fresh/.buildkite_monitor/db_demo/with_logs_table/'

LAST_2_HOURS = (datetime.utcnow() - pd.Timedelta(hours=2)).strftime('%Y-%m-%dT%H:%M')


builds_bronze_path = PATH_TO_TABLES + 'bronze_tables/builds.parquet'
jobs_bronze_path = PATH_TO_TABLES + 'bronze_tables/jobs.parquet'
agents_bronze_path = PATH_TO_TABLES + 'bronze_tables/agents.parquet'
job_logs_bronze_path = PATH_TO_TABLES + 'bronze_tables/job_logs.parquet'



 

# silver layer
jobs_silver_path = PATH_TO_TABLES + 'silver_tables/jobs.parquet'
builds_silver_path = PATH_TO_TABLES + 'silver_tables/builds.parquet'


In [59]:
params = {
    'created_from': LAST_2_HOURS,
    'per_page': 100,
    'include_retried_jobs': True
}

In [60]:
raw = fetch_data(params, BUILDKITE_API_TOKEN, ORGANIZATION_SLUG, PIPELINE_SLUG)
if raw.empty:
    print('No data to process')
    raise SystemExit

now_utc = pd.Timestamp.now(tz='UTC')
raw['timestamp'] = now_utc

now = datetime.now(zoneinfo.ZoneInfo('Europe/Helsinki')).strftime('%Y-%m-%d_%H:%M')
raw.drop(['pipeline.steps'], axis=1).to_parquet(PATH_TO_TABLES + f'raw_data/raw_data_fetch_{now}.parquet', index=False)


In [61]:
builds_useful_columns = ['timestamp', 'url', 'id', 'web_url', 'number', 'state', 'jobs']
jobs_useful_columns = ['build_url', 'id', 'name', 'state',  'web_url', 'soft_failed', 'created_at', 'scheduled_at', 'runnable_at', 'started_at', 'finished_at', 'expired_at', 'retried', 'retried_in_job_id', 'retries_count', 'retry_source',
       'retry_type', 'agent.id', 'agent.name', 'agent.web_url', 'agent.connection_state', 'agent.meta_data', 'log_url']


In [12]:
raw_data_dir = 'raw_data/'
parquet_files = glob.glob(os.path.join(raw_data_dir, 'logurls*.parquet'))
parquet_files.sort(key=os.path.getmtime, reverse=False)
parquet_files

['raw_data/logurls_2024-12-17T12:55:58.945829.parquet',
 'raw_data/logurls_2024-12-18T07:03:35.337742.parquet']

In [15]:
from db_functions import fetch_job_log

In [68]:
builds_bronze_path = PATH_TO_TABLES + 'bronze_tables/builds.parquet'
jobs_bronze_path = PATH_TO_TABLES + 'bronze_tables/jobs.parquet'
agents_bronze_path = PATH_TO_TABLES + 'bronze_tables/agents.parquet'
job_logs_bronze_path = PATH_TO_TABLES + 'bronze_tables/job_logs.parquet'
job_logs_silver_path = PATH_TO_TABLES + 'silver_tables/job_logs.parquet'

raw_data_dir = 'raw_data/'
 

# silver layer
jobs_silver_path = PATH_TO_TABLES + 'silver_tables/jobs.parquet'
builds_silver_path = PATH_TO_TABLES + 'silver_tables/builds.parquet'


#df = pd.read_parquet(job_logs_bronze_path)

def check_parquet_files(raw_data_dir, job_logs_bronze_path):
    # Get all parquet files in the raw_data directory
    parquet_files = glob.glob(os.path.join(raw_data_dir, 'logurls_*.parquet'))
    
    # Extract timestamps from filenames
    file_timestamps = [os.path.basename(f).split('_')[1].replace('.parquet', '').replace('T', ' ') for f in parquet_files]
    
    # Read the job_logs_bronze table
    job_logs_bronze = pd.read_parquet(job_logs_bronze_path)
    
    # Convert the timestamp column to string format
    job_logs_timestamps = job_logs_bronze['timestamp'].astype('str')
    
    # Check if all file timestamps are present in the job_logs_bronze table
    missing_timestamps = [ts for ts in file_timestamps if ts not in job_logs_timestamps.values]
    
    if missing_timestamps:
        print("Missing timestamps in job_logs_bronze table:", missing_timestamps)
    else:
        print("All parquet files are accounted for in the job_logs_bronze table.")
    return missing_timestamps  

def find_new_jobs(df, bronze):
    print('bronze file size', bronze.shape)
    merged_job_logs = pd.merge(bronze, df, on=['id', 'state', 'log_url'], how='right', indicator=True)
    print('merged_job_logs: ',merged_job_logs.shape)
    #display(merged_job_logs.head())
    df = merged_job_logs[merged_job_logs['_merge'] == 'right_only']
    print(f'jobs in new raw file that are not in bronze: {df.shape}')
    df = df[['timestamp_y', 'id', 'state', 'log_url']].rename(columns={'timestamp_y': 'timestamp'})
    #display(df.head())  
    return df  

def download_logs_for_df(df, BUILDKITE_API_TOKEN, ORGANIZATION_SLUG, PIPELINE_SLUG):
    start_time = time.time()
    for idx in df.id.unique():
        if df[df.id==idx].log_url.notna().values[0]:
            log, rate_limit_remaining, rate_limit_reset = fetch_job_log(df[df.id==idx].log_url.values[0], BUILDKITE_API_TOKEN, ORGANIZATION_SLUG, PIPELINE_SLUG)
            df.loc[df.id==idx, 'log'] = log
            rate_limit = int(rate_limit_remaining)
            #print(rate_limit)
            if rate_limit < 30:
                print(f'rate_limit is low = {rate_limit}, sleeping for {rate_limit_reset} seconds')
                time.sleep(int(rate_limit_reset))

    end_time = time.time()

    # Calculate elapsed time
    elapsed_time = end_time - start_time
    print(f"Elapsed time: {elapsed_time} seconds")
    return df

bronze_job_logs_exists = os.path.exists(job_logs_bronze_path)

if not bronze_job_logs_exists:
    print("Bronze job logs table doesn't exist")
    parquet_files = glob.glob(os.path.join(raw_data_dir, 'logurls*.parquet'))
    parquet_files.sort(key=os.path.getmtime, reverse=False)
    #bronze_not_empty = False

else:
    print('bronze table with job logs exists')
    missing_ts = check_parquet_files(raw_data_dir, job_logs_bronze_path)
    if missing_ts:        
        parquet_files = [raw_data_dir + f"/logurls_{ts}.parquet".replace(' ', 'T') for ts in missing_ts.sort_values(by='timestamp')]
        print("the following parquet files haven't been read yet:", parquet_files) 
    else:
        print('all parquet files have been read already, exiting')
        raise SystemExit

    



for logurl_parq in parquet_files:
    print(logurl_parq)
    df = pd.read_parquet(logurl_parq)
    print('parquet file size', df.shape)
    df['log'] = None

    if bronze_job_logs_exists:
        print('bronze not empty')
        bronze = pd.read_parquet(job_logs_bronze_path)

        df = find_new_jobs(df, bronze)

    df = download_logs_for_df(df, BUILDKITE_API_TOKEN, ORGANIZATION_SLUG, PIPELINE_SLUG)        
    
    # makes sense to save only jobs that have not null log, but it then will save these jobs in raw since it is not yet in bronze, if logurl 
    # is null then it will be filtered out in logs_fetching, but if logurl is not null, then log will be fetched.
    # as first iteration I probably better save everything even if logurl is None, will think about it later.

    print(f'concatinating df {df.shape} to bronze parquet')
    if bronze_job_logs_exists:
        bronze = pd.concat([bronze, df], axis=0).sort_values(by='timestamp')
        bronze.to_parquet(job_logs_bronze_path)
    else:    
        df.to_parquet(job_logs_bronze_path) 
        bronze_job_logs_exists = True
    
    #display(df.head())

Bronze job logs table doesn't exist
raw_data/logurls_2024-12-17T12:55:58.945829.parquet
parquet file size (92, 4)


Elapsed time: 21.92045283317566 seconds
concatinating df (92, 5) to bronze parquet
raw_data/logurls_2024-12-18T07:03:35.337742.parquet
parquet file size (113, 4)
bronze not empty
bronze file size (92, 5)
merged_job_logs:  (113, 8)
jobs in new raw file that are not in bronze: (113, 8)
Elapsed time: 22.52284598350525 seconds
concatinating df (113, 5) to bronze parquet


In [51]:


def check_parquet_files(raw_data_dir, job_logs_bronze_path):
    # Get all parquet files in the raw_data directory
    parquet_files = glob.glob(os.path.join(raw_data_dir, 'logurls_*.parquet'))
    
    # Extract timestamps from filenames
    file_timestamps = [os.path.basename(f).split('_')[1].replace('.parquet', '').replace('T', ' ') for f in parquet_files]
    
    # Read the job_logs_bronze table
    job_logs_bronze = pd.read_parquet(job_logs_bronze_path)
    
    # Convert the timestamp column to string format
    job_logs_timestamps = job_logs_bronze['timestamp'].astype('str')
    
    # Check if all file timestamps are present in the job_logs_bronze table
    missing_timestamps = [ts for ts in file_timestamps if ts not in job_logs_timestamps.values]
    
    if missing_timestamps:
        print("Missing timestamps in job_logs_bronze table:", missing_timestamps)
    else:
        print("All parquet files are accounted for in the job_logs_bronze table.")
    return missing_timestamps    


# Usage
raw_data_dir = 'raw_data'
job_logs_bronze_path = 'bronze_tables/job_logs.parquet'
missing_ts = check_parquet_files(raw_data_dir, job_logs_bronze_path)

if missing_ts:
    print('Missing timestamps:', missing_ts)
else:
    print('no missing ts')

Missing timestamps in job_logs_bronze table: ['2024-12-18 07:03:35.337742', '2024-12-17 12:55:58.945829']
Missing timestamps: ['2024-12-18 07:03:35.337742', '2024-12-17 12:55:58.945829']


In [ ]:
bronze_not_empty = False
for logurl_parq in parquet_files:
    print(logurl_parq)
    df = pd.read_parquet(logurl_parq)
    print('parquet file size', df.shape)
    df['log'] = None

    if bronze_not_empty:
        print('bronze not empty')
        bronze = pd.read_parquet(job_logs_bronze_path)
        print('bronze file size', bronze.shape)
        merged_job_logs = pd.merge(bronze, df, on=['id', 'state', 'log_url'], how='right', indicator=True)
        print('merged_job_logs: ',merged_job_logs.shape)
        display(merged_job_logs.head())
        df = merged_job_logs[merged_job_logs['_merge'] == 'right_only']
        print(f'jobs in new raw file that are not in bronze: {df.shape}')
        df = df[['timestamp_y', 'id', 'state', 'log_url']].rename(columns={'timestamp_y': 'timestamp'})
        display(df.head())

            
    start_time = time.time()
    
    for idx in df.id.unique():
        if df[df.id==idx].log_url.notna().values[0]:
            log, rate_limit_remaining, rate_limit_reset = fetch_job_log(df[df.id==idx].log_url.values[0], BUILDKITE_API_TOKEN, ORGANIZATION_SLUG, PIPELINE_SLUG)
            df.loc[df.id==idx, 'log'] = log
            rate_limit = int(rate_limit_remaining)
            #print(rate_limit)
            if rate_limit < 30:
                print(f'rate_limit is low = {rate_limit}, sleeping for {rate_limit_reset} seconds')
                time.sleep(int(rate_limit_reset))

    end_time = time.time()

    # Calculate elapsed time
    elapsed_time = end_time - start_time
    print(f"Elapsed time: {elapsed_time} seconds")
    # makes sense to save only jobs that have not null log, but it then will save these jobs in raw since it is not yet in bronze, if logurl 
    # is null then it will be filtered out in logs_fetching, but if logurl is not null, then log will be fetched.
    # as first iteration I probably better save everything even if logurl is None, will think about it later.

    print(f'concating df {df.shape} to bronze parquet')
    if bronze_not_empty:
        bronze = pd.concat([bronze, df], axis=0)
        bronze.to_parquet(job_logs_bronze_path)
    else:    
        df.to_parquet(job_logs_bronze_path) 
        bronze_not_empty = True
    
    display(df.head())

    

raw_data/logurls_2024-12-17T12:55:58.945829.parquet
parquet file size (92, 4)


Elapsed time: 22.559311389923096 seconds
concating df (92, 5) to bronze parquet


,timestamp,id,state,log_url,log
0,2024-12-17 12:55:58.945829,0193d45a-b421-4c37-a13d-2402ef75de28,passed,https://api.buildkite.com/v2/organizations/vll...,~~~ Running global environment hook\n[2024-12-...
1,2024-12-17 12:55:58.945829,0193d45a-e8a9-4069-9aad-e1305393eddf,passed,https://api.buildkite.com/v2/organizations/vll...,~~~ Running global environment hook\n[2024-12-...
2,2024-12-17 12:55:58.945829,0193d45a-e8b2-4fcd-97f5-64112713496d,passed,https://api.buildkite.com/v2/organizations/vll...,~~~ Running global environment hook\n[2024-12-...
3,2024-12-17 12:55:58.945829,0193d45a-e8b6-4231-8c77-9bcc368a4a51,passed,https://api.buildkite.com/v2/organizations/vll...,~~~ Running global environment hook\n[2024-12-...
4,2024-12-17 12:55:58.945829,0193d45a-e8bb-46f5-b224-fe428a7f5e61,blocked,None,None


raw_data/logurls_2024-12-18T07:03:35.337742.parquet
parquet file size (113, 4)
bronze not empty
bronze file size (92, 5)
merged_job_logs:  (113, 8)


,timestamp_x,id,state,log_url,log_x,timestamp_y,log_y,_merge
0,NaT,0193d881-3d04-49a0-aa73-6d44d0d54fb2,passed,https://api.buildkite.com/v2/organizations/vll...,NaN,2024-12-18 07:03:35.337742,None,right_only
1,NaT,0193d881-6d20-4cc4-a34c-ed89b9f18a79,blocked,None,NaN,2024-12-18 07:03:35.337742,None,right_only
2,NaT,0193d881-6d20-4d32-bf52-b8b09ade9a69,blocked,https://api.buildkite.com/v2/organizations/vll...,NaN,2024-12-18 07:03:35.337742,None,right_only
3,NaT,0193d881-6d22-4662-aacc-6ac4b9298b23,blocked,None,NaN,2024-12-18 07:03:35.337742,None,right_only
4,NaT,0193d881-6d23-426b-9e37-b98d4bda522c,blocked,https://api.buildkite.com/v2/organizations/vll...,NaN,2024-12-18 07:03:35.337742,None,right_only


jobs in new raw file that are not in bronze: (113, 8)


,timestamp,id,state,log_url
0,2024-12-18 07:03:35.337742,0193d881-3d04-49a0-aa73-6d44d0d54fb2,passed,https://api.buildkite.com/v2/organizations/vll...
1,2024-12-18 07:03:35.337742,0193d881-6d20-4cc4-a34c-ed89b9f18a79,blocked,None
2,2024-12-18 07:03:35.337742,0193d881-6d20-4d32-bf52-b8b09ade9a69,blocked,https://api.buildkite.com/v2/organizations/vll...
3,2024-12-18 07:03:35.337742,0193d881-6d22-4662-aacc-6ac4b9298b23,blocked,None
4,2024-12-18 07:03:35.337742,0193d881-6d23-426b-9e37-b98d4bda522c,blocked,https://api.buildkite.com/v2/organizations/vll...


Elapsed time: 22.05575156211853 seconds
concating df (113, 5) to bronze parquet


,timestamp,id,state,log_url,log
0,2024-12-18 07:03:35.337742,0193d881-3d04-49a0-aa73-6d44d0d54fb2,passed,https://api.buildkite.com/v2/organizations/vll...,~~~ Running global environment hook\n[2024-12-...
1,2024-12-18 07:03:35.337742,0193d881-6d20-4cc4-a34c-ed89b9f18a79,blocked,None,NaN
2,2024-12-18 07:03:35.337742,0193d881-6d20-4d32-bf52-b8b09ade9a69,blocked,https://api.buildkite.com/v2/organizations/vll...,
3,2024-12-18 07:03:35.337742,0193d881-6d22-4662-aacc-6ac4b9298b23,blocked,None,NaN
4,2024-12-18 07:03:35.337742,0193d881-6d23-426b-9e37-b98d4bda522c,blocked,https://api.buildkite.com/v2/organizations/vll...,


In [49]:
missing_ts = check_parquet_files(raw_data_dir, job_logs_bronze_path)


All parquet files are accounted for in the job_logs_bronze table.


In [52]:
missing_ts

['2024-12-18 07:03:35.337742', '2024-12-17 12:55:58.945829']

In [57]:
parquet_files = [raw_data_dir + f"/logurls_{ts}.parquet".replace(' ', 'T') for ts in missing_ts] 
parquet_files

['raw_data/logurls_2024-12-18T07:03:35.337742.parquet',
 'raw_data/logurls_2024-12-17T12:55:58.945829.parquet']

In [69]:
q = pd.read_parquet(job_logs_bronze_path)
print(q.shape)

(205, 5)


In [ ]:
df = raw.copy()
df = types_fix(df[builds_useful_columns])

jobs = pd.json_normalize(df['jobs'].explode())
jobs = types_fix(jobs[jobs_useful_columns], jobs=True)
jobs['timestamp'] = df['timestamp'].values[0]


# Builds bronze table
df.drop(['jobs'], axis=1, inplace=True)
bronze_builds = bronze_builds(df, builds_bronze_path, False)

Bronze builds table exists, checking what is updated
There are new builds, appending them to the existing data
There were changes, write new bronze data to parquet


In [6]:
jobs.shape

(252, 24)

In [ ]:
#finished jobs: passed, failed, blocked, canceled

In [33]:
sl_b = pd.read_parquet(PATH_TO_TABLES + 'bronze_tables/job_logs.parquet')
display(sl_b.head())
print(sl_b.shape)

,timestamp,id,state,log_url,log
0,2024-12-17 12:55:58.945829,0193d45a-b421-4c37-a13d-2402ef75de28,passed,https://api.buildkite.com/v2/organizations/vll...,~~~ Running global environment hook\n[2024-12-...
1,2024-12-17 12:55:58.945829,0193d45a-e8a9-4069-9aad-e1305393eddf,passed,https://api.buildkite.com/v2/organizations/vll...,~~~ Running global environment hook\n[2024-12-...
2,2024-12-17 12:55:58.945829,0193d45a-e8b2-4fcd-97f5-64112713496d,passed,https://api.buildkite.com/v2/organizations/vll...,~~~ Running global environment hook\n[2024-12-...
3,2024-12-17 12:55:58.945829,0193d45a-e8b6-4231-8c77-9bcc368a4a51,passed,https://api.buildkite.com/v2/organizations/vll...,~~~ Running global environment hook\n[2024-12-...
4,2024-12-17 12:55:58.945829,0193d45a-e8bb-46f5-b224-fe428a7f5e61,blocked,None,None


(205, 5)


In [37]:
sl_b.timestamp.astype('str').values[0]

'2024-12-17 12:55:58.945829'

In [7]:
sl = pd.read_parquet(PATH_TO_TABLES + 'silver_tables/job_logs.parquet')
display(sl.head())
print(sl.shape)

,timestamp,id,state,log_url,log
0,2024-12-13 11:01:23.548490,0193bf9a-bc9b-4a1b-b931-ca5157f3c3ca,passed,https://api.buildkite.com/v2/organizations/vll...,~~~ Running global environment hook\n[2024-12-...
37,2024-12-13 11:01:23.548490,0193bf9b-7ca1-4707-8dad-4fc64e42aa8d,blocked,None,None
38,2024-12-13 11:01:23.548490,0193bf9b-7ca2-491a-b731-6cade5184c43,blocked,https://api.buildkite.com/v2/organizations/vll...,
40,2024-12-13 11:01:23.548490,0193bf9b-7ca4-4885-ab5d-08cf70a0260c,blocked,None,None
41,2024-12-13 11:01:23.548490,0193bf9b-7ca4-4a33-8842-6c64bcf385a1,blocked,https://api.buildkite.com/v2/organizations/vll...,


(293, 5)


I need to save job ids that are in a finalized state, but where do I save it in separate parquet file with timestamp in the name? and then logs fetchin script would read these files one after another. Need a mechanism how logs fetching script decides that it read all records from previous batch and now it needs to read another batch.

Reads parquet file from raw job logs from 12.00, adds job ids to bronze/silver table, fetches the logs.
then log fetching script is launched again, it reads the latest timestamp in the table?

In [8]:
def bronze_jobs_ids_for_logs(df, jobs_bronze_path, job_logs_bronze_path, token, org_slug, pipe_slug, testing):
   if not os.path.exists(jobs_bronze_path):
      print("Bronze jobs table doesn't exist")
      df.to_parquet(jobs_bronze_path, index=False)
      job_logurls = df[df.state.isin(['passed', 'failed', 'blocked', 'canceled'])][['timestamp', 'id','state', 'log_url']]
      timest = job_logurls.timestamp.values[0]
      print('timestamp to use in logurls:', timest)
      job_logurls.to_parquet(PATH_TO_TABLES + f'raw_data/logurls_{timest}.parquet', index=False)
      return df, job_logurls
   else: 
      print("Bronze jobs table exists, checking what is updated")
      bronze = pd.read_parquet(jobs_bronze_path) 

      # filter for latest timestamps for each job
      latest_indices = bronze.groupby(['id'])['timestamp'].idxmax()
      bronze_latest = bronze[bronze.index.isin(latest_indices)]
      
      existing_jobs = pd.merge(bronze_latest, df, on='id', how='inner', suffixes=('_old', ''))
      update_check = False
      if not existing_jobs.empty:
         print('info about jobs exists in jobs bronze table')
         comparison_columns_old = [col for col in existing_jobs.columns if '_old' in col and 'timestamp' not in col and 'agent_meta_data' not in col]
         comparison_columns_new = [col[:-4] for col in comparison_columns_old]
         df_old = existing_jobs[comparison_columns_old].copy()#.sort_index(axis=1).copy()
         df_old_columns = [col[:-4] for col in df_old.columns]
         df_old.columns = df_old_columns
         df_old.sort_index(axis=1, inplace=True)
         df_new = existing_jobs[comparison_columns_new].sort_index(axis=1).copy()    
         assert (df_old.columns == df_new.columns).all()
         mask = df_old.fillna('None').ne(df_new.fillna('None'))
         updated_rows = existing_jobs[mask.any(axis=1)]
         if not updated_rows.empty:
               print('There are updated rows')
               columns_without_old_suffix = [col for col in updated_rows.columns if not col.endswith('_old')]
               update_check = True
                  
               bronze = pd.concat([bronze, updated_rows[columns_without_old_suffix]], ignore_index=True)  
               

      # Identify new jobs
      new_jobs = df[(~df['id'].isin(bronze['id']))]
      print('New jobs')
      #display(new_jobs)
      if not new_jobs.empty:
      # Append new jobs to the existing data
         print('There are new jobs, appending them to the existing data')
         update_check = True
         #print('bronze jobs with new rows:')
         bronze = pd.concat([bronze, new_jobs], ignore_index=True)
         #display(bronze.head())


      if update_check:
         print('There were changes, write new bronze data to parquet')
         if not testing:
            bronze.to_parquet(jobs_bronze_path, index=False) 


      # handle job logs
      # filter for latest timestamps for each job log
      bronze_job_logs = pd.read_parquet(job_logs_bronze_path) 

      job_logurls = df[df.state.isin(['passed', 'failed', 'blocked', 'canceled'])][['timestamp', 'id','state', 'log_url']]
      job_logs_latest_indices = bronze_job_logs.groupby(['id'])['timestamp'].idxmax()
      bronze_job_logs_latest = bronze_job_logs[bronze_job_logs.index.isin(job_logs_latest_indices)]
      print('bronze job logs shape latest timestamps only, should be (293, 5):', bronze_job_logs_latest.shape)

      merged_job_logs = pd.merge(bronze_job_logs_latest, job_logurls, on=['id', 'state'], how='right', indicator=True)
      print('merged_job_logs: ',merged_job_logs.shape)
      display(merged_job_logs.head())
      new_job_logurls = merged_job_logs[merged_job_logs['_merge'] == 'right_only']
      new_job_logurls = new_job_logurls[['timestamp_y', 'id', 'state', 'log_url_y']].rename(columns={'timestamp_y': 'timestamp', 'log_url_y': 'log_url'})

      

      if not new_job_logurls.empty:
         print('new job logs shape:', new_job_logurls.shape)
         display(new_job_logurls.head())

         timest = new_job_logurls.timestamp.values[0]
         print('timestamp to use in logurls:', timest)
         new_job_logurls.to_parquet(PATH_TO_TABLES + f'raw_data/logurls_{timest}.parquet', index=False)
         print('new job logurls saved')
      else:
         print('There are no new job logurls')   

      return bronze

In [9]:
b = bronze_jobs_ids_for_logs(jobs, jobs_bronze_path, job_logs_bronze_path, BUILDKITE_API_TOKEN, ORGANIZATION_SLUG, PIPELINE_SLUG, False)


Bronze jobs table exists, checking what is updated
New jobs
There are new jobs, appending them to the existing data
There were changes, write new bronze data to parquet


bronze job logs shape latest timestamps only, should be (293, 5): (293, 5)
merged_job_logs:  (113, 8)


,timestamp_x,id,state,log_url_x,log,timestamp_y,log_url_y,_merge
0,NaT,0193d881-3d04-49a0-aa73-6d44d0d54fb2,passed,NaN,NaN,2024-12-18 07:03:35.337742,https://api.buildkite.com/v2/organizations/vll...,right_only
1,NaT,0193d881-6d20-4cc4-a34c-ed89b9f18a79,blocked,NaN,NaN,2024-12-18 07:03:35.337742,NaN,right_only
2,NaT,0193d881-6d20-4d32-bf52-b8b09ade9a69,blocked,NaN,NaN,2024-12-18 07:03:35.337742,https://api.buildkite.com/v2/organizations/vll...,right_only
3,NaT,0193d881-6d22-4662-aacc-6ac4b9298b23,blocked,NaN,NaN,2024-12-18 07:03:35.337742,NaN,right_only
4,NaT,0193d881-6d23-426b-9e37-b98d4bda522c,blocked,NaN,NaN,2024-12-18 07:03:35.337742,https://api.buildkite.com/v2/organizations/vll...,right_only


new job logs shape: (113, 4)


,timestamp,id,state,log_url
0,2024-12-18 07:03:35.337742,0193d881-3d04-49a0-aa73-6d44d0d54fb2,passed,https://api.buildkite.com/v2/organizations/vll...
1,2024-12-18 07:03:35.337742,0193d881-6d20-4cc4-a34c-ed89b9f18a79,blocked,NaN
2,2024-12-18 07:03:35.337742,0193d881-6d20-4d32-bf52-b8b09ade9a69,blocked,https://api.buildkite.com/v2/organizations/vll...
3,2024-12-18 07:03:35.337742,0193d881-6d22-4662-aacc-6ac4b9298b23,blocked,NaN
4,2024-12-18 07:03:35.337742,0193d881-6d23-426b-9e37-b98d4bda522c,blocked,https://api.buildkite.com/v2/organizations/vll...


timestamp to use in logurls: 2024-12-18T07:03:35.337742
new job logurls saved


table doesn't exist:
1. fetch data
2. write table to parquet, save ids in file
3. another container maybe reads the file and fetches logs for these ids and writes into the table: job_id, log

tables exists:
1. fetch data
2. compare new data to old: all columns except the logs
3. save ids of new data to file
4. new ids and updated info for old ids save into the table
5. another container when finished doing previous job, takes new portion of ids and downloads logs for them
6. compare newly downloaded logs to older existing ones, if there is change, save it (again problem to make this comparison to work properly)
7. logs of jobs that do not exist yet, save them into the log table


Logs have unix time timestamps, Theoretically I can parse it for each job one line in the table with its timestamp. It can be on demand function. Info from log table can be joined to jobs table by id. 

1 part of the script writes everything except the logs, proceeds to silver table and to alerts table, sends alerts if needed.
2 part of the script downloads logs and writes to log table on its own pace.


In the existing bronze table I can delete log column, it will probably reduce the size significantly.

logfile has errors about connection, maybe it doesn't write logs correctly somehow and that's why comparison is not working.